# RECIPE GEOFISIKA: ANALISIS DATA SEISMIK REFRAKSI

**Nama:** Indira Nazwa Nandita 

**NIM:** H061231054 

**Kelas:** KECERDASAN DAN ARTIFISIAL UNTUK GEOFISIKA (B)

---

## Recipe ID: GEO-REFRAKSI-001

**Bidang:** Geofisika near-surface dan seismologi  
**Topik:** Pengolahan data waktu tempuh (*travel time*) untuk mengestimasi kecepatan gelombang P (\(V_P\)) dan menginterpretasi model lapisan bawah permukaan dengan metode seismik refraksi.

> **Petunjuk notebook:** Sel bertipe *Markdown* berisi penjelasan/laporan. Sel bertipe *Code* berisi kode Python yang dijalankan dengan tombol **Run**. Jalankan sel secara berurutan dari atas ke bawah.

**Catatan:** Notebook ini menggunakan data simulasi agar dapat dijalankan tanpa data lapangan. Jika memiliki data hasil picking, ganti pengaturan sumber data dan siapkan CSV dengan kolom `shot_id`, `geophone_x_m`, dan `travel_time_s`. Nilai simulasi hanya untuk demonstrasi, bukan hasil pengukuran lapangan.

## 1. Problem — Permasalahan

Metode seismik refraksi memanfaatkan waktu tempuh gelombang seismik dari sumber menuju geofon. Pada kondisi tertentu, gelombang yang dibiaskan secara kritis di sepanjang batas lapisan berkecepatan lebih tinggi dapat kembali terdeteksi di permukaan. Hubungan waktu tempuh dengan jarak sumber–geofon digunakan untuk memperkirakan kecepatan gelombang dan karakteristik lapisan bawah permukaan.

Tujuan recipe ini adalah:

1. Memeriksa struktur dan kualitas dasar data waktu tempuh.
2. Memvisualisasikan waktu tempuh terhadap jarak sumber–geofon.
3. Mengidentifikasi segmen linear awal secara sederhana.
4. Mengestimasi kecepatan semu dari kemiringan kurva waktu tempuh.
5. Mengilustrasikan estimasi dua lapisan menggunakan model refraksi ideal.
6. Menyimpan hasil dan membuat grafik.

**Batasan:** Pemilihan segmen otomatis pada notebook ini hanya demonstrasi. Interpretasi yang dapat dipertanggungjawabkan membutuhkan picking first-break yang diperiksa, geometri survei yang benar, evaluasi beberapa posisi sumber (*shot points*), serta pertimbangan topografi dan heterogenitas bawah permukaan.

## 2. Minimal Theory — Teori Singkat

Pada kurva waktu tempuh terhadap jarak, kemiringan segmen linear berhubungan dengan kelambatan (*slowness*):

\[
t = t_i + \frac{x}{V}
\]

sehingga:

\[
V = \frac{1}{m}
\]

dengan \(m\) adalah kemiringan garis \(t\)-\(x\) dalam satuan s/m, \(V\) adalah kecepatan gelombang dalam m/s, dan \(t_i\) adalah intersep waktu.

Untuk model ideal dua lapisan horizontal dengan \(V_2 > V_1\), waktu tempuh gelombang langsung dan gelombang refraksi kepala dapat ditulis:

\[
t_{\mathrm{langsung}} = \frac{x}{V_1}
\]

\[
t_{\mathrm{refraksi}} = t_i + \frac{x}{V_2}
\]

Intersep waktu ideal memenuhi:

\[
t_i = \frac{2h\cos i_c}{V_1}, \qquad
\sin i_c = \frac{V_1}{V_2}
\]

Maka estimasi kedalaman antarmuka:

\[
h = \frac{t_i V_1}{2\cos i_c}
\]

Keterangan:
- \(x\): jarak sumber–geofon (m).
- \(t\): waktu tempuh (s).
- \(V_1, V_2\): kecepatan lapisan pertama dan kedua (m/s).
- \(t_i\): intersep waktu (s).
- \(i_c\): sudut kritis yang didefinisikan terhadap garis normal.
- \(h\): ketebalan lapisan pertama (m).

**Catatan metodologis:** Rumus dua lapisan ini berlaku untuk model sederhana dengan lapisan relatif horizontal, lapisan kedua lebih cepat, dan asumsi gelombang refraksi kepala. Model nyata dapat membutuhkan metode interpretasi yang lebih lengkap.

## 3. Persiapan — Library yang Digunakan

Kode berikut mengimpor pustaka untuk pengolahan tabel, perhitungan numerik, regresi linear, dan visualisasi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Library berhasil diimpor.")

## 4. Prompt — Instruksi Analisis untuk AI

Gunakan instruksi berikut sebagai panduan ketika meminta AI membantu menganalisis data:

> Saya sedang mengolah data seismik refraksi. Periksa kelengkapan data, satuan jarak dan waktu, serta konsistensi geometri sumber–geofon. Buat grafik waktu tempuh terhadap jarak, bantu identifikasi segmen linear first-arrival, estimasi kecepatan dari kemiringan regresi, dan jika asumsi model dua lapisan terpenuhi, hitung estimasi awal kedalaman antarmuka. Buat grafik dan ekspor hasil ke CSV. Jangan mengarang hasil pengukuran. Jika data simulasi digunakan, nyatakan secara jelas bahwa hasilnya bukan data lapangan.

## 5. Code — Pengaturan Sumber Data

Pilih sumber data:

- `USE_SIMULATED_DATA = True`: membuat data simulasi untuk memastikan notebook bisa langsung dijalankan.
- `USE_SIMULATED_DATA = False`: membaca file `refraksi.csv` dari folder kerja notebook.

Untuk data picking asli, CSV harus memiliki kolom:
- `shot_id`: identitas sumber/shot.
- `geophone_x_m`: koordinat atau jarak geofon terhadap sumber dalam meter. Untuk workflow sederhana ini, gunakan jarak sumber–geofon nonnegatif.
- `travel_time_s`: waktu tempuh first-break dalam detik.

In [ ]:
USE_SIMULATED_DATA = True

if USE_SIMULATED_DATA:
    # DATA SIMULASI untuk demonstrasi, bukan hasil lapangan.
    rng = np.random.default_rng(42)

    V1_true = 500.0   # m/s, kecepatan lapisan atas simulasi
    V2_true = 1800.0  # m/s, kecepatan lapisan bawah simulasi
    h_true = 8.0      # m, ketebalan lapisan atas simulasi

    x = np.linspace(2, 80, 40)
    ic = np.arcsin(V1_true / V2_true)
    ti = 2 * h_true * np.cos(ic) / V1_true
    crossover = ti / (1 / V1_true - 1 / V2_true)

    t_direct = x / V1_true
    t_refracted = ti + x / V2_true
    travel_time = np.minimum(t_direct, t_refracted)
    travel_time += rng.normal(0, 0.0008, size=len(x))
    travel_time = np.maximum(travel_time, 0)

    df = pd.DataFrame({
        "shot_id": "SHOT_1",
        "geophone_x_m": x,
        "travel_time_s": travel_time
    })
    print("Menggunakan DATA SIMULASI. Hasil bukan hasil pengukuran lapangan.")
    print(f"Parameter model simulasi: V1={V1_true:.0f} m/s, "
          f"V2={V2_true:.0f} m/s, h={h_true:.1f} m.")
else:
    input_path = Path("refraksi.csv")
    if not input_path.exists():
        raise FileNotFoundError(
            "File refraksi.csv tidak ditemukan. Letakkan file di folder notebook "
            "atau ubah input_path ke lokasi file yang benar."
        )
    df = pd.read_csv(input_path)
    print(f"Membaca data dari: {input_path.resolve()}")

df.head()

## 6. Code — Pemeriksaan Dasar Data

Bagian ini memeriksa kolom wajib, nilai kosong, nilai numerik, serta nilai jarak dan waktu yang masuk akal. Periksa pula bahwa data yang dianalisis berasal dari shot yang sama.

In [ ]:
required_columns = ["shot_id", "geophone_x_m", "travel_time_s"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Kolom berikut tidak ditemukan: {missing_columns}")

df = df[required_columns].copy()
df["geophone_x_m"] = pd.to_numeric(df["geophone_x_m"], errors="coerce")
df["travel_time_s"] = pd.to_numeric(df["travel_time_s"], errors="coerce")

print("Jumlah baris dan kolom:", df.shape)
print("\nJumlah nilai kosong per kolom:")
print(df.isna().sum())

if df.isna().any().any():
    raise ValueError("Data mengandung nilai kosong/non-numerik. Bersihkan data sebelum analisis.")

if len(df) < 5:
    raise ValueError("Data terlalu sedikit untuk regresi linear sederhana.")

if (df["geophone_x_m"] < 0).any():
    raise ValueError("Jarak harus nonnegatif untuk workflow sederhana ini.")

if (df["travel_time_s"] < 0).any():
    raise ValueError("Waktu tempuh tidak boleh negatif.")

shot_ids = df["shot_id"].astype(str).unique()
if len(shot_ids) != 1:
    raise ValueError(
        "Notebook ini menganalisis satu shot pada satu waktu. "
        "Filter data agar hanya satu shot_id yang diproses."
    )

df = df.sort_values("geophone_x_m").reset_index(drop=True)

if df["geophone_x_m"].duplicated().any():
    print("Peringatan: terdapat jarak geofon duplikat; periksa geometri atau data.")

print(f"\nShot yang dianalisis: {shot_ids[0]}")
print(f"Rentang jarak: {df['geophone_x_m'].min():.2f}–{df['geophone_x_m'].max():.2f} m")
print(f"Rentang waktu tempuh: {df['travel_time_s'].min():.5f}–{df['travel_time_s'].max():.5f} s")
display(df.head())

## 7. Code — Visualisasi Kurva Waktu Tempuh

Grafik waktu tempuh terhadap jarak merupakan langkah awal untuk mengenali perubahan kemiringan. Perubahan kemiringan dapat mengindikasikan kedatangan gelombang dengan kecepatan berbeda, tetapi tidak boleh ditafsirkan sebagai batas lapisan tanpa pemeriksaan lebih lanjut.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(
    df["geophone_x_m"], df["travel_time_s"] * 1000,
    label="Picking first-break", marker="o"
)
plt.xlabel("Jarak sumber–geofon (m)")
plt.ylabel("Waktu tempuh (ms)")
plt.title(f"Kurva waktu tempuh seismik refraksi — {shot_ids[0]}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Code — Estimasi Kecepatan dari Segmen Linear

Untuk latihan ini, data dibagi menjadi dua bagian berdasarkan jarak: bagian jarak dekat dan bagian jarak jauh. Regresi linear dilakukan secara terpisah. Pembagian otomatis ini **bukan** pengganti picking atau pemilihan segmen oleh interpreter; hasil harus diperiksa pada grafik dan disesuaikan dengan data sebenarnya.

In [ ]:
x = df["geophone_x_m"].to_numpy(dtype=float)
t = df["travel_time_s"].to_numpy(dtype=float)

split_index = max(3, min(len(df) - 3, len(df) // 2))

# Segmen jarak dekat dan jauh untuk demonstrasi.
near_mask = np.arange(len(df)) < split_index
far_mask = ~near_mask

def fit_travel_time_segment(x_seg, t_seg, label):
    if len(x_seg) < 3:
        raise ValueError(f"Segmen {label} memiliki kurang dari 3 titik.")
    slope, intercept = np.polyfit(x_seg, t_seg, 1)
    if slope <= 0:
        raise ValueError(
            f"Kemiringan segmen {label} tidak positif. Periksa data dan pemilihan segmen."
        )
    velocity = 1.0 / slope
    predicted = slope * x_seg + intercept
    residual = t_seg - predicted
    ss_res = np.sum(residual**2)
    ss_tot = np.sum((t_seg - np.mean(t_seg))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {
        "segment": label,
        "slope_s_per_m": slope,
        "intercept_s": intercept,
        "velocity_m_per_s": velocity,
        "R2": r2
    }

fit_near = fit_travel_time_segment(x[near_mask], t[near_mask], "Jarak dekat")
fit_far = fit_travel_time_segment(x[far_mask], t[far_mask], "Jarak jauh")

fits = pd.DataFrame([fit_near, fit_far])
display(fits)

print("Interpretasi awal:")
print(f"Kecepatan semu segmen jarak dekat: {fit_near['velocity_m_per_s']:.1f} m/s")
print(f"Kecepatan semu segmen jarak jauh: {fit_far['velocity_m_per_s']:.1f} m/s")
print("Periksa kembali segmen regresi; pembagian otomatis dapat mencampur jenis kedatangan.")

## 9. Code — Grafik Regresi dan Estimasi Kedalaman Dua Lapisan

Estimasi kedalaman hanya dihitung jika hasil kecepatan memenuhi asumsi sederhana \(V_2 > V_1 > 0\), dan intersep waktu segmen jauh positif. Kecepatan hasil regresi jarak dekat diperlakukan sebagai \(V_1\), sedangkan regresi jarak jauh sebagai \(V_2\). Jika asumsi ini tidak terpenuhi, kedalaman tidak dihitung.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(x, t * 1000, label="Data picking", zorder=3)

x_near = x[near_mask]
x_far = x[far_mask]

plt.plot(
    x_near, (fit_near["slope_s_per_m"] * x_near + fit_near["intercept_s"]) * 1000,
    label=f"Regresi dekat: V={fit_near['velocity_m_per_s']:.0f} m/s"
)
plt.plot(
    x_far, (fit_far["slope_s_per_m"] * x_far + fit_far["intercept_s"]) * 1000,
    label=f"Regresi jauh: V={fit_far['velocity_m_per_s']:.0f} m/s"
)
plt.xlabel("Jarak sumber–geofon (m)")
plt.ylabel("Waktu tempuh (ms)")
plt.title("Kurva waktu tempuh dan regresi linear")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

V1 = fit_near["velocity_m_per_s"]
V2 = fit_far["velocity_m_per_s"]
ti = fit_far["intercept_s"]

if V2 > V1 > 0 and ti > 0 and V1 / V2 < 1:
    ic = np.arcsin(V1 / V2)
    h_est = ti * V1 / (2 * np.cos(ic))
    crossover_est = ti / (1 / V1 - 1 / V2)

    print("=== ESTIMASI AWAL MODEL DUA LAPISAN ===")
    print(f"V1 (kecepatan lapisan atas): {V1:.2f} m/s")
    print(f"V2 (kecepatan lapisan bawah): {V2:.2f} m/s")
    print(f"Intersep waktu ti: {ti:.6f} s")
    print(f"Sudut kritis ic: {np.degrees(ic):.2f} derajat")
    print(f"Estimasi ketebalan lapisan atas h: {h_est:.2f} m")
    print(f"Estimasi jarak silang: {crossover_est:.2f} m")
else:
    h_est = np.nan
    crossover_est = np.nan
    print(
        "Kedalaman tidak dihitung karena asumsi model dua lapisan tidak terpenuhi "
        "(perlu V2 > V1 dan intersep waktu positif)."
    )

print("\nPENTING: angka di atas adalah estimasi otomatis awal, bukan interpretasi final.")

## 10. Code — Menyimpan Hasil ke CSV

Hasil picking yang telah diperiksa, ringkasan regresi, dan estimasi model dua lapisan disimpan ke file CSV di folder kerja notebook.

In [ ]:
df.to_csv("data_refraksi_terurut.csv", index=False)
fits.to_csv("hasil_regresi_refraksi.csv", index=False)

summary = pd.DataFrame([{
    "shot_id": str(shot_ids[0]),
    "V1_m_per_s": V1,
    "V2_m_per_s": V2,
    "intercept_time_s": ti,
    "estimated_upper_layer_thickness_m": h_est,
    "estimated_crossover_distance_m": crossover_est,
    "data_type": "simulasi" if USE_SIMULATED_DATA else "pengukuran",
    "note": "Estimasi awal; validasi pemilihan segmen dan asumsi model diperlukan."
}])
summary.to_csv("ringkasan_refraksi.csv", index=False)

print("File tersimpan:")
print("- data_refraksi_terurut.csv")
print("- hasil_regresi_refraksi.csv")
print("- ringkasan_refraksi.csv")
display(summary)

## 11. Result — Hasil dan Interpretasi

Setelah seluruh sel kode dijalankan, perhatikan:

- Grafik waktu tempuh (*travel-time curve*).
- Perubahan kemiringan kurva, jika terlihat.
- Kecepatan semu dari segmen jarak dekat dan jarak jauh.
- Nilai \(R^2\) sebagai salah satu indikator kecocokan regresi, bukan satu-satunya ukuran kualitas interpretasi.
- Estimasi ketebalan lapisan atas jika asumsi model dua lapisan terpenuhi.
- File CSV yang berisi data terurut, hasil regresi, dan ringkasan estimasi.

**Jika menggunakan data simulasi:** hasil hanya menunjukkan bahwa alur komputasi dapat dijalankan. Nilainya tidak boleh dilaporkan sebagai hasil pengukuran lapangan dan tidak dapat digunakan untuk menyimpulkan kondisi geologi lokasi nyata.

**Jika menggunakan data lapangan:** interpretasikan hasil berdasarkan first-break yang telah diperiksa, konfigurasi geofon, posisi sumber, topografi, jumlah shot, serta kondisi geologi. Model dua lapisan sederhana dapat memberikan hasil bias jika terdapat lapisan berkecepatan rendah di bawah lapisan berkecepatan tinggi, lapisan miring, atau variasi lateral yang kuat.

## 12. Optional Check — Checklist Kualitas

Gunakan daftar periksa berikut sebelum memakai hasil dalam laporan.

- Kolom data lengkap dan tidak memiliki nilai kosong.
- Satuan jarak dan waktu telah dipastikan.
- Data yang dibandingkan berasal dari shot yang sama dan geometri survei benar.
- First-break diperiksa terhadap rekaman seismik, bukan hanya diproses otomatis.
- Grafik waktu tempuh diperiksa untuk outlier dan perubahan kemiringan.
- Segmen regresi dipilih sesuai cabang kedatangan fisik yang masuk akal.
- Kecepatan hasil interpretasi positif dan konsisten dengan asumsi model.
- Data simulasi dan data pengukuran dibedakan secara eksplisit.
- Interpretasi didukung informasi geologi atau data geofisika lain jika tersedia.

## 13. Kesimpulan

Recipe GEO-REFRAKSI-001 menyusun alur dasar pengolahan data seismik refraksi, mulai dari pemeriksaan data, visualisasi kurva waktu tempuh, regresi linear, estimasi kecepatan gelombang, hingga ilustrasi perhitungan kedalaman antarmuka untuk model dua lapisan ideal. Notebook ini dapat digunakan sebagai latihan pengolahan yang dapat diulang (*reproducible workflow*). Namun, pemilihan first-break dan segmen linear harus divalidasi sebelum hasil digunakan untuk interpretasi geofisika.